In [1]:
# @title
import os
import sys
import time

# Built-in imports
import warnings
from collections import Counter
import pickle

# Data manipulation
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
import missingno

# Sklearn imports
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    root_mean_squared_error,
    mean_absolute_error
)

# ML Models - Linear
from sklearn.linear_model import (
    LogisticRegression,
    Perceptron,
    SGDClassifier,
    Lasso,
    LassoCV
)

from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, precision_score, recall_score, f1_score

# @title
from math import sqrt
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.preprocessing import LabelEncoder
import pickle

# Disable warnings
warnings.filterwarnings('ignore')

In [2]:
from google.colab import drive
drive.mount("/content/drive/", force_remount=True)

Mounted at /content/drive/


# Load Lasso Models And Scaler

In [3]:
def load_model_and_scaler(model_path):
    """
    Charge un modèle et un scaler à partir du chemin donné.

    Args:
        model_path (str): Chemin du dossier contenant le modèle et le scaler.

    Returns:
        tuple: Le modèle et le scaler chargés.
    """
    with open(os.path.join(model_path, 'lasso_globale_model.pkl'), 'rb') as f:
        model = pickle.load(f)
    with open(os.path.join(model_path, 'lasso_globale_scaler.pkl'), 'rb') as f:
        scaler = pickle.load(f)
    return model, scaler

In [4]:
directory = "/content/drive/MyDrive/Memoire/DIORES/Models/V4/LassoGlobal/L1MPI"

predictors = []
for i in range(1, 4):
    model_path = os.path.join(directory, f"Doc{i}")
    predictor, scaler = load_model_and_scaler(model_path)
    predictors.append(predictor)

# Evaluate Predictions

In [5]:
def evaluate_predictionsV2(df_result, rank_method='average'):
    df = df_result.copy()
    df['Rang_L1_New'] = df['Score L1'].rank(ascending=False, method=rank_method)
    df['Rang_Predit'] = df['Score_Predit'].rank(ascending=False, method=rank_method)

    # Métriques de base pour l'erreur de classement
    rmse = np.sqrt(mean_squared_error(df['Rang_L1_New'], df['Rang_Predit']))
    mae = mean_absolute_error(df['Rang_L1_New'], df['Rang_Predit'])

    try:
        r2 = r2_score(df['Rang_L1_New'], df['Rang_Predit'])
    except:
        r2 = float('nan')

    # MRR comme avant
    df_strict = df.loc[df['RESULTAT'] == 'PASSE']
    df_strict['mrr'] = 1 / df_strict['Rang_Predit'] if len(df_strict) > 0 else 0
    mrr_strict = df_strict['mrr'].sum() if len(df_strict) > 0 else 0

    df_open = df.loc[df['RESULTAT'] != 'NON ADMIS']
    df_open['mrr'] = 1 / df_open['Rang_Predit'] if len(df_open) > 0 else 0
    mrr_open = df_open['mrr'].sum() if len(df_open) > 0 else 0

    # Nouvelles métriques pour la classification
    if 'Prediction_Status' in df.columns and 'RESULTAT' in df.columns:
        df['Actual_Admission'] = df['RESULTAT'].apply(lambda x: 1 if x != 'NON ADMIS' else 0)
        df['Predicted_Admission'] = df['Prediction_Status'].apply(lambda x: 1 if x != 'NON ADMIS' else 0)

        precision = precision_score(df['Actual_Admission'], df['Predicted_Admission'], zero_division=0)
        recall = recall_score(df['Actual_Admission'], df['Predicted_Admission'], zero_division=0)
        f1 = f1_score(df['Actual_Admission'], df['Predicted_Admission'], zero_division=0)
    else:
        precision, recall, f1 = 0, 0, 0

    # Métriques d'équité basées sur les colonnes disponibles
    equity_index_gender = 0
    equity_index_region = 0

    # Équité par genre
    if 'Homme' in df.columns and 'Femme' in df.columns:
        df['Gender'] = df.apply(lambda x: 'Homme' if x['Homme'] == 1 else 'Femme', axis=1)
        try:
            gender_errors = df.groupby('Gender').apply(lambda x: np.sqrt(mean_squared_error(x['Rang_L1_New'], x['Rang_Predit'])))
            if len(gender_errors) > 1:
                equity_index_gender = np.std(gender_errors)
        except:
            pass

    # Équité par région de naissance
    if 'REGION_DE_NAISSANCE' in df.columns:
        try:
            region_errors = df.groupby('REGION_DE_NAISSANCE').apply(
                lambda x: np.sqrt(mean_squared_error(x['Rang_L1_New'], x['Rang_Predit']))
                if len(x) >= 5 else None
            ).dropna()
            if len(region_errors) > 1:
                equity_index_region = np.std(region_errors)
        except:
            pass

    return {
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'mrr_strict': mrr_strict,
        'mrr_open': mrr_open,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'equity_index_gender': equity_index_gender,
        'equity_index_region': equity_index_region
    }

In [6]:
def evaluate_all_predictionsV2(df_results, suffix, rank_method='average'):
    print(f"\nEVALUATIONS POUR {suffix}")

    # Initialiser un dictionnaire pour stocker les métriques agrégées
    aggregate_metrics = {}

    for idx, df_result in enumerate(df_results):
        metrics = evaluate_predictionsV2(df_result, rank_method)

        # Ajouter ou accumuler les métriques
        for key, value in metrics.items():
            if key in aggregate_metrics:
                aggregate_metrics[key] += value
            else:
                aggregate_metrics[key] = value

    # Calculer les moyennes
    for key in aggregate_metrics:
        aggregate_metrics[key] /= len(df_results)
        print(f"{key}_{suffix}_{rank_method}: {aggregate_metrics[key]:.4f}")

    return aggregate_metrics

In [7]:
def evaluateV2(docs, predictors, suffix='', rank_method='average'):
  results = []
  for i in range(len(docs)):
      print(f"\n{suffix} Doc{i+1}")
      # Prédiction
      df_result = predictors[i].predict(docs[i].copy())

      # S'assurer que les colonnes de rang existent
      if 'Rang_L1_New' not in df_result.columns and 'Score L1' in df_result.columns:
          df_result['Rang_L1_New'] = df_result['Score L1'].rank(ascending=False, method=rank_method)

      if 'Rang_Predit' not in df_result.columns and 'Score_Predit' in df_result.columns:
          df_result['Rang_Predit'] = df_result['Score_Predit'].rank(ascending=False, method=rank_method)

      results.append(df_result)

  # Calculer les métriques globales
  metrics = evaluate_all_predictionsV2(results, suffix, rank_method)

  # Analyse simple par dimension démographique
  print("\nAnalyse simplifiée de l'équité:")

  # Par genre
  print("\nRMSE par genre:")
  for i, result in enumerate(results):
      if 'Homme' in result.columns and 'Femme' in result.columns:
          result['Gender'] = result.apply(lambda x: 'Homme' if x['Homme'] == 1 else 'Femme', axis=1)
          for gender in ['Homme', 'Femme']:
              try:
                  gender_data = result[result['Gender'] == gender]
                  if len(gender_data) >= 5:
                      rmse = np.sqrt(((gender_data['Rang_L1_New'] - gender_data['Rang_Predit'])**2).mean())
                      print(f"  Doc{i+1}, {gender}: RMSE = {rmse:.4f} (n={len(gender_data)})")
              except Exception as e:
                  print(f"  Erreur: {str(e)}")

  return metrics

# Class Predictors

## DioresPredictorLasso

In [8]:
class DioresPredictorLasso:
    def __init__(self, model, scaler, features):
        self.model = model
        self.scaler = scaler
        self.features = features

    def predict(self, df):
        """
        Fait des prédictions sur un DataFrame et renvoie les résultats au format attendu.
        """
        # Créer une copie du DataFrame pour ne pas modifier l'original
        df_result = df.copy()

        # Vérifier les colonnes manquantes
        missing_cols = set(self.features) - set(df.columns)
        if missing_cols:
            print(f"Attention: Colonnes manquantes: {missing_cols}")

            raise ValueError(f"Colonnes manquantes dans le DataFrame: {missing_cols}")

        # Sélectionner et standardiser les features
        try:
            X = df_result[list(self.features)]
            X_scaled = self.scaler.transform(X)

            # Faire les prédictions
            predictions = self.model.predict(X_scaled)

            # Ajouter les prédictions au DataFrame résultat
            df_result['Score_Predit'] = predictions

            return df_result
        except Exception as e:
            print(f"Erreur lors de la prédiction: {str(e)}")
            print(f"Features attendues: {self.features}")
            print(f"Colonnes disponibles: {df_result.columns.tolist()}")
            raise

## DioresEnsembliste

In [9]:
class DioresEnsembliste(BaseEstimator, ClassifierMixin):
    """
    Modèle ensembliste hiérarchique pour la prédiction des résultats étudiants.

    Architecture en arbre :
    1. Admission (NON ADMIS/AUTORISE/PASSE)
    2. Si admis → Session (Deuxième/Première)
    3. Si Première Session → Mention (Passable/Assez-Bien/Bien/Très-Bien)

    Résultat final : dictionnaire avec toutes les prédictions
    """

    def __init__(self, model_admission=None, model_session=None, model_mention=None, feature_columns=None):
        """
        Initialise le modèle ensembliste

        Args:
            model_admission: modèle pour prédire l'admission
            model_session: modèle pour prédire la session
            model_mention: modèle pour prédire la mention
            feature_columns: liste des colonnes de features à utiliser
        """
        self.model_admission = model_admission
        self.model_session = model_session
        self.model_mention = model_mention
        self.feature_columns = feature_columns or []

        # Mappings pour convertir les prédictions numériques en labels
        self.admission_labels = {0: 'NON ADMIS', 1: 'AUTORISE/PASSE'}
        self.session_labels = {0: 'Deuxième Session', 1: 'Première Session'}
        self.mention_labels = {0: 'Passable', 1: 'Assez-Bien/Bien/Très-Bien'}

    def load_models_from_files(self, model_paths, feature_columns):
        """
        Charge les modèles depuis des fichiers pickle

        Args:
            model_paths: dict avec les chemins des modèles
                        {'admission': 'path1.pkl', 'session': 'path2.pkl', 'mention': 'path3.pkl'}
            feature_columns: liste des colonnes de features à utiliser
        """
        self.feature_columns = feature_columns

        if 'admission' in model_paths:
            with open(model_paths['admission'], 'rb') as f:
                self.model_admission = pickle.load(f)

        if 'session' in model_paths:
            with open(model_paths['session'], 'rb') as f:
                self.model_session = pickle.load(f)

        if 'mention' in model_paths:
            with open(model_paths['mention'], 'rb') as f:
                self.model_mention = pickle.load(f)

        print("Modèles chargés avec succès")
        return self

    def fit(self, X, y):
        """
        Méthode fit (requise par BaseEstimator)
        Dans notre cas, les modèles sont déjà entraînés
        """
        # Les modèles individuels sont déjà entraînés
        # Cette méthode est juste pour la compatibilité sklearn
        return self

    def predict_single(self, x):
        """
        Prédit le parcours complet d'un seul étudiant

        Args:
            x: features d'un étudiant (array 1D)

        Returns:
            dict: résultat complet du parcours
        """
        result = {
            'admission': None,
            'session': None,
            'mention': None,
            'final_status': None,
            'path': []
        }

        # Reshape pour avoir la bonne forme (1, n_features)
        x_reshaped = x.reshape(1, -1)

        # Étape 1: Prédiction de l'admission
        if self.model_admission is not None:
            admission_pred = self.model_admission.predict(x_reshaped)[0]
            admission_label = self.admission_labels[admission_pred]
            result['admission'] = admission_label
            result['path'].append(f"Admission: {admission_label}")

            # Si NON ADMIS, on s'arrête ici
            if admission_pred == 0:  # NON ADMIS
                result['final_status'] = 'NON ADMIS'
                result['path'].append("ARRÊT: Non admis")
                return result

            # Si AUTORISE/PASSE, on continue avec la session
            if self.model_session is not None:
                session_pred = self.model_session.predict(x_reshaped)[0]
                session_label = self.session_labels[session_pred]
                result['session'] = session_label
                result['path'].append(f"Session: {session_label}")

                # Si Deuxième Session, on s'arrête ici
                if session_pred == 0:  # Deuxième Session
                    result['final_status'] = f"AUTORISE - {session_label}"
                    result['path'].append("ARRÊT: Deuxième session")
                    return result

                # Si Première Session, on continue avec la mention
                if self.model_mention is not None:
                    mention_pred = self.model_mention.predict(x_reshaped)[0]
                    mention_label = self.mention_labels[mention_pred]
                    result['mention'] = mention_label
                    result['path'].append(f"Mention: {mention_label}")
                    result['final_status'] = f"AUTORISE - Première Session - {mention_label}"
                    result['path'].append("ARRÊT: Parcours complet")
                else:
                    result['final_status'] = f"AUTORISE - {session_label}"
                    result['path'].append("ARRÊT: Pas de modèle mention")
            else:
                result['final_status'] = admission_label
                result['path'].append("ARRÊT: Pas de modèle session")
        else:
            result['final_status'] = "ERREUR: Pas de modèle admission"
            result['path'].append("ERREUR: Pas de modèle admission")

        return result

    def predict(self, df):
        """
        Méthode compatible avec evaluateV2
        Fait des prédictions et retourne un DataFrame avec Score_Predit

        Args:
            df: DataFrame avec les features des étudiants

        Returns:
            DataFrame: copie du DataFrame original avec colonnes ajoutées:
                      - Score_Predit: score numérique pour le ranking
                      - Prediction_Status: statut prédit (pour classification)
                      - Admission_Pred, Session_Pred, Mention_Pred: prédictions détaillées
        """
        # Créer une copie du DataFrame
        df_result = df.copy()

        # Vérifier les colonnes manquantes
        missing_cols = set(self.feature_columns) - set(df.columns)
        if missing_cols:
            raise ValueError(f"Colonnes manquantes dans le DataFrame: {missing_cols}")

        # Extraire les features
        X = df_result[self.feature_columns].values

        # Faire les prédictions hiérarchiques
        hierarchical_results = self.predict_hierarchical(X)

        # Convertir en scores et statuts
        scores = []
        statuses = []
        admission_preds = []
        session_preds = []
        mention_preds = []

        for result in hierarchical_results:
            # Calculer le score basé sur le parcours hiérarchique
            score = self._calculate_score_from_result(result)
            scores.append(score)
            statuses.append(result['final_status'])
            admission_preds.append(result['admission'])
            session_preds.append(result['session'])
            mention_preds.append(result['mention'])

        # Ajouter les colonnes au DataFrame
        df_result['Score_Predit'] = scores
        df_result['Prediction_Status'] = statuses
        df_result['Admission_Pred'] = admission_preds
        df_result['Session_Pred'] = session_preds
        df_result['Mention_Pred'] = mention_preds

        return df_result

    def _calculate_score_from_result(self, result):
        """
        Convertit un résultat hiérarchique en score numérique
        Plus le score est élevé, meilleur est le résultat

        Args:
            result: dictionnaire de résultat de predict_single

        Returns:
            float: score numérique
        """
        base_score = 0

        # Score basé sur l'admission (0-40 points)
        if result['admission'] == 'NON ADMIS':
            base_score += 0
        else:  # AUTORISE/PASSE
            base_score += 40

            # Score basé sur la session (0-30 points supplémentaires)
            if result['session'] == 'Deuxième Session':
                base_score += 10
            elif result['session'] == 'Première Session':
                base_score += 30

                # Score basé sur la mention (0-30 points supplémentaires)
                if result['mention'] == 'Passable':
                    base_score += 10
                elif result['mention'] == 'Assez-Bien/Bien/Très-Bien':
                    base_score += 30

        # Ajouter un petit bruit aléatoire pour éviter les égalités parfaites
        # (important pour le ranking)
        noise = np.random.uniform(-0.1, 0.1)

        return base_score + noise

    def predict_hierarchical(self, X):
        """
        Alias pour predict() (version hiérarchique originale)

        Args:
            X: features des étudiants (array 2D)

        Returns:
            list: liste des résultats hiérarchiques pour chaque étudiant
        """
        results = []
        for i in range(len(X)):
            result = self.predict_single(X[i])
            results.append(result)

        return results

    def predict_final_status(self, X):
        """
        Retourne seulement le statut final pour chaque étudiant

        Args:
            X: features des étudiants

        Returns:
            list: liste des statuts finaux
        """
        results = self.predict(X)
        return [result['final_status'] for result in results]

    def predict_admission_only(self, X):
        """
        Retourne seulement les prédictions d'admission (compatible sklearn)

        Args:
            X: features des étudiants

        Returns:
            array: prédictions d'admission (0 ou 1)
        """
        if self.model_admission is None:
            raise ValueError("Modèle d'admission non chargé")

        return self.model_admission.predict(X)

    def analyze_student_path(self, X, student_index=0):
        """
        Analyse détaillée du parcours d'un étudiant spécifique

        Args:
            X: features des étudiants
            student_index: index de l'étudiant à analyser
        """
        result = self.predict_single(X[student_index])

        print(f"ANALYSE DE L'ÉTUDIANT {student_index}")
        print("="*50)

        for step in result['path']:
            print(f"• {step}")

        print(f"\nSTATUT FINAL: {result['final_status']}")

        return result

    def get_statistics(self, X):
        """
        Calcule des statistiques sur les prédictions

        Args:
            X: features des étudiants

        Returns:
            dict: statistiques détaillées
        """
        results = self.predict(X)

        stats = {
            'total_students': len(results),
            'non_admis': 0,
            'admis_deuxieme_session': 0,
            'admis_premiere_session': 0,
            'mentions': {'Passable': 0, 'Assez-Bien/Bien/Très-Bien': 0}
        }

        for result in results:
            if 'NON ADMIS' in result['final_status']:
                stats['non_admis'] += 1
            elif 'Deuxième Session' in result['final_status']:
                stats['admis_deuxieme_session'] += 1
            elif 'Première Session' in result['final_status']:
                stats['admis_premiere_session'] += 1

                if result['mention']:
                    if 'Passable' in result['mention']:
                        stats['mentions']['Passable'] += 1
                    else:
                        stats['mentions']['Assez-Bien/Bien/Très-Bien'] += 1

        # Calculer les pourcentages
        total = stats['total_students']
        stats['percentages'] = {
            'non_admis': (stats['non_admis'] / total) * 100,
            'admis_deuxieme_session': (stats['admis_deuxieme_session'] / total) * 100,
            'admis_premiere_session': (stats['admis_premiere_session'] / total) * 100
        }

        return stats

    def print_statistics(self, X):
        """Affiche les statistiques de manière lisible"""
        stats = self.get_statistics(X)

        print("STATISTIQUES DES PRÉDICTIONS")
        print("="*40)
        print(f"Total étudiants: {stats['total_students']}")
        print(f"Non admis: {stats['non_admis']} ({stats['percentages']['non_admis']:.1f}%)")
        print(f"Admis 2ème session: {stats['admis_deuxieme_session']} ({stats['percentages']['admis_deuxieme_session']:.1f}%)")
        print(f"Admis 1ère session: {stats['admis_premiere_session']} ({stats['percentages']['admis_premiere_session']:.1f}%)")

        if stats['admis_premiere_session'] > 0:
            print("\nMentions (1ère session):")
            print(f"  Passable: {stats['mentions']['Passable']}")
            print(f"  Assez-Bien/Bien/Très-Bien: {stats['mentions']['Assez-Bien/Bien/Très-Bien']}")

## DioresHybridPredictor

In [10]:
class DioresHybridPredictor(BaseEstimator, ClassifierMixin):
    """
    Modèle hybride qui combine DioresEnsembliste + DioresPredictorLasso

    Processus en 2 étapes :
    1. DioresEnsembliste : Classification hiérarchique (Admission → Session → Mention)
    2. DioresPredictorLasso : Affinage du score avec régression Lasso

    Le score final combine les deux approches pour un meilleur ranking
    """

    def __init__(self,
                 model_admission=None, model_session=None, model_mention=None,
                 lasso_model=None, lasso_scaler=None,
                 feature_columns=None, lasso_features=None):
        """
        Initialise le modèle hybride

        Args:
            model_admission, model_session, model_mention: modèles de classification
            lasso_model: modèle Lasso pour l'affinage
            lasso_scaler: scaler pour le modèle Lasso
            feature_columns: colonnes pour les modèles de classification
            lasso_features: colonnes pour le modèle Lasso
        """
        # Modèles de classification hiérarchique
        self.model_admission = model_admission
        self.model_session = model_session
        self.model_mention = model_mention
        self.feature_columns = feature_columns or []

        # Modèle Lasso
        self.lasso_model = lasso_model
        self.lasso_scaler = lasso_scaler
        self.lasso_features = lasso_features or []

        # Mappings pour les classifications
        self.admission_labels = {0: 'NON ADMIS', 1: 'AUTORISE/PASSE'}
        self.session_labels = {0: 'Deuxième Session', 1: 'Première Session'}
        self.mention_labels = {0: 'Passable', 1: 'Assez-Bien/Bien/Très-Bien'}

        # Poids pour combiner les scores (ajustables)
        self.ensemble_weight = 0.4  # Poids du score ensembliste
        self.lasso_weight = 0.6     # Poids du score Lasso

    def load_models_from_files(self, ensemble_paths, lasso_path, feature_columns, lasso_features):
        """
        Charge tous les modèles depuis des fichiers

        Args:
            ensemble_paths: dict avec chemins des modèles de classification
            lasso_path: chemin vers le dossier du modèle Lasso
            feature_columns: colonnes pour les modèles de classification
            lasso_features: colonnes pour le modèle Lasso
        """
        self.feature_columns = feature_columns
        # S'assurer que lasso_features est une liste
        self.lasso_features = list(lasso_features) if lasso_features else []

        # Charger les modèles de classification
        if 'admission' in ensemble_paths:
            with open(ensemble_paths['admission'], 'rb') as f:
                self.model_admission = pickle.load(f)

        if 'session' in ensemble_paths:
            with open(ensemble_paths['session'], 'rb') as f:
                self.model_session = pickle.load(f)

        if 'mention' in ensemble_paths:
            with open(ensemble_paths['mention'], 'rb') as f:
                self.model_mention = pickle.load(f)

        # Charger le modèle Lasso et son scaler
        import os
        lasso_model_path = os.path.join(lasso_path, 'lasso_globale_model.pkl')
        lasso_scaler_path = os.path.join(lasso_path, 'lasso_globale_scaler.pkl')

        if os.path.exists(lasso_model_path):
            with open(lasso_model_path, 'rb') as f:
                self.lasso_model = pickle.load(f)

        if os.path.exists(lasso_scaler_path):
            with open(lasso_scaler_path, 'rb') as f:
                self.lasso_scaler = pickle.load(f)

        print("Tous les modèles chargés avec succès")
        print(f"Features Lasso (type: {type(self.lasso_features)}): {self.lasso_features}")
        return self

    def predict_single_ensemble(self, x):
        """
        Étape 1 : Prédiction hiérarchique avec l'ensembliste
        """
        result = {
            'admission': None,
            'session': None,
            'mention': None,
            'final_status': None
        }

        # Reshape pour avoir la bonne forme
        x_reshaped = x.reshape(1, -1)

        # Prédiction admission
        if self.model_admission is not None:
            admission_pred = self.model_admission.predict(x_reshaped)[0]
            admission_label = self.admission_labels[admission_pred]
            result['admission'] = admission_label

            if admission_pred == 0:  # NON ADMIS
                result['final_status'] = 'NON ADMIS'
                return result

            # Prédiction session
            if self.model_session is not None:
                session_pred = self.model_session.predict(x_reshaped)[0]
                session_label = self.session_labels[session_pred]
                result['session'] = session_label

                if session_pred == 0:  # Deuxième Session
                    result['final_status'] = f"AUTORISE - {session_label}"
                    return result

                # Prédiction mention
                if self.model_mention is not None:
                    mention_pred = self.model_mention.predict(x_reshaped)[0]
                    mention_label = self.mention_labels[mention_pred]
                    result['mention'] = mention_label
                    result['final_status'] = f"AUTORISE - Première Session - {mention_label}"
                else:
                    result['final_status'] = f"AUTORISE - {session_label}"
            else:
                result['final_status'] = admission_label

        return result

    def predict_single_lasso(self, x):
        """
        Étape 2 : Prédiction avec le modèle Lasso
        """
        if self.lasso_model is None or self.lasso_scaler is None:
            return 0.0

        try:
            # Reshape et standardiser
            x_reshaped = x.reshape(1, -1)
            x_scaled = self.lasso_scaler.transform(x_reshaped)

            # Prédiction Lasso
            lasso_score = self.lasso_model.predict(x_scaled)[0]
            return lasso_score
        except Exception as e:
            print(f"Erreur Lasso: {e}")
            return 0.0

    def calculate_ensemble_score(self, ensemble_result):
        """
        Convertit le résultat ensembliste en score numérique
        """
        base_score = 0

        if ensemble_result['admission'] == 'NON ADMIS':
            base_score = 0
        else:  # AUTORISE/PASSE
            base_score = 40

            if ensemble_result['session'] == 'Deuxième Session':
                base_score += 10
            elif ensemble_result['session'] == 'Première Session':
                base_score += 30

                if ensemble_result['mention'] == 'Passable':
                    base_score += 10
                elif ensemble_result['mention'] == 'Assez-Bien/Bien/Très-Bien':
                    base_score += 30

        return base_score

    def predict_single_hybrid(self, x_ensemble, x_lasso):
        """
        Prédiction hybride complète pour un étudiant

        Args:
            x_ensemble: features pour les modèles de classification
            x_lasso: features pour le modèle Lasso

        Returns:
            dict: résultat hybride complet
        """
        # Étape 1: Prédiction ensembliste
        ensemble_result = self.predict_single_ensemble(x_ensemble)
        ensemble_score = self.calculate_ensemble_score(ensemble_result)

        # Étape 2: Prédiction Lasso
        lasso_score = self.predict_single_lasso(x_lasso)

        # Étape 3: Combinaison des scores
        # Normaliser le score Lasso pour qu'il soit dans une plage similaire
        normalized_lasso = lasso_score  # Ajustez selon vos données

        # Score final pondéré
        final_score = (self.ensemble_weight * ensemble_score +
                      self.lasso_weight * normalized_lasso)

        # Résultat hybride
        hybrid_result = {
            **ensemble_result,  # Inclut admission, session, mention, final_status
            'ensemble_score': ensemble_score,
            'lasso_score': lasso_score,
            'final_hybrid_score': final_score
        }

        return hybrid_result

    def predict(self, df):
        """
        Méthode principale compatible avec evaluateV2

        Args:
            df: DataFrame avec toutes les features nécessaires

        Returns:
            DataFrame: DataFrame avec colonnes de prédiction ajoutées
        """
        df_result = df.copy()

        # Vérifier les colonnes manquantes pour l'ensembliste
        missing_ensemble = set(self.feature_columns) - set(df.columns)
        if missing_ensemble:
            raise ValueError(f"Colonnes manquantes pour l'ensembliste: {missing_ensemble}")

        # Vérifier les colonnes manquantes pour Lasso
        missing_lasso = set(self.lasso_features) - set(df.columns)
        available_lasso = set(self.lasso_features) & set(df.columns)

        use_lasso = True
        if missing_lasso:
            print(f"ATTENTION: Colonnes manquantes pour Lasso: {missing_lasso}")
            print(f"Colonnes disponibles pour Lasso: {len(available_lasso)}/{len(self.lasso_features)}")

            if len(available_lasso) < len(self.lasso_features) * 0.8:  # Si moins de 80% des colonnes
                print("Trop de colonnes manquantes pour Lasso. Utilisation de l'ensembliste uniquement.")
                use_lasso = False
            else:
                print("Utilisation des colonnes Lasso disponibles uniquement.")
                # Utiliser seulement les colonnes disponibles
                self.lasso_features = list(available_lasso)

        # Debug: Afficher les colonnes utilisées
        print(f"Colonnes DataFrame: {df.columns.tolist()}")
        print(f"Colonnes Lasso demandées: {self.lasso_features}")

        # Extraire les features
        X_ensemble = df_result[self.feature_columns].values
        X_lasso = None

        if use_lasso:
            try:
                X_lasso = df_result[self.lasso_features].values
                print(f"Features Lasso extraites avec succès: {X_lasso.shape}")
            except Exception as e:
                print(f"Erreur lors de l'extraction des features Lasso: {e}")
                use_lasso = False

        # Prédictions hybrides
        scores = []
        statuses = []
        ensemble_scores = []
        lasso_scores = []
        admission_preds = []
        session_preds = []
        mention_preds = []

        for i in range(len(X_ensemble)):
            if use_lasso and X_lasso is not None:
                try:
                    result = self.predict_single_hybrid(X_ensemble[i], X_lasso[i])
                    scores.append(result['final_hybrid_score'])
                    lasso_scores.append(result['lasso_score'])
                except Exception as e:
                    print(f"Erreur hybride pour l'étudiant {i}: {e}")
                    # Fallback sur ensembliste seulement
                    ensemble_result = self.predict_single_ensemble(X_ensemble[i])
                    ensemble_score = self.calculate_ensemble_score(ensemble_result)
                    result = {**ensemble_result, 'ensemble_score': ensemble_score}
                    scores.append(ensemble_score)
                    lasso_scores.append(0.0)
            else:
                # Utiliser seulement l'ensembliste
                ensemble_result = self.predict_single_ensemble(X_ensemble[i])
                ensemble_score = self.calculate_ensemble_score(ensemble_result)
                result = {**ensemble_result, 'ensemble_score': ensemble_score}
                scores.append(ensemble_score)
                lasso_scores.append(0.0)

            statuses.append(result['final_status'])
            ensemble_scores.append(result.get('ensemble_score', 0))
            admission_preds.append(result['admission'])
            session_preds.append(result['session'])
            mention_preds.append(result['mention'])

        # Ajouter toutes les colonnes de résultats
        df_result['Score_Predit'] = scores
        df_result['Prediction_Status'] = statuses
        df_result['Ensemble_Score'] = ensemble_scores
        df_result['Lasso_Score'] = lasso_scores
        df_result['Admission_Pred'] = admission_preds
        df_result['Session_Pred'] = session_preds
        df_result['Mention_Pred'] = mention_preds

        return df_result

    def fit(self, X, y):
        """Méthode fit requise par BaseEstimator (les modèles sont pré-entraînés)"""
        return self

    def set_weights(self, ensemble_weight, lasso_weight):
        """
        Ajuste les poids de combinaison des scores

        Args:
            ensemble_weight: poids du score ensembliste (0-1)
            lasso_weight: poids du score Lasso (0-1)
        """
        total = ensemble_weight + lasso_weight
        self.ensemble_weight = ensemble_weight / total
        self.lasso_weight = lasso_weight / total
        print(f"Nouveaux poids - Ensemble: {self.ensemble_weight:.2f}, Lasso: {self.lasso_weight:.2f}")

## DAP Predictor

In [11]:
class DAP_predictor:
  def __init__(self):
    pass

  def predict(self, X):
    df = X.copy()
    df['Score_Predit'] = df['Score DAP']
    return df

# FEATURE COLUMNS

In [12]:
FEATURE_COLUMNS = ['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
                  'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
                  'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
                  'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
                  'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
                  'Ets. de provenance_Encode', 'Centre d\'Ec._Encode',
                  'Académie de l\'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode']

# MPI TEST

In [13]:
doc1_df_test = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc1/doc1_df_test.csv");
doc2_df_test = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc2/doc2_df_test.csv");
doc3_df_test = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc3/doc3_df_test.csv");

docs = []
docs.append( doc1_df_test)
docs.append( doc2_df_test)
docs.append( doc3_df_test)


model_paths_doc1 = {
    'admission': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc1/admi_non_admi_best_model.pkl',
    'session': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc1/session_best_model.pkl',
    'mention': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc1/mention_best_model.pkl'
}

model_paths_doc2 = {
    'admission': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc2/admi_non_admi_best_model.pkl',
    'session': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc2/session_best_model.pkl',
    'mention': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc2/mention_best_model.pkl'
}

model_paths_doc3 = {
    'admission': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc3/admi_non_admi_best_model.pkl',
    'session': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc3/session_best_model.pkl',
    'mention': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc3/mention_best_model.pkl'
}


lasso_path_doc1 = "/content/drive/MyDrive/Memoire/DIORES/Models/V4/LassoGlobal/L1MPI/Doc1"
lasso_path_doc2 = "/content/drive/MyDrive/Memoire/DIORES/Models/V4/LassoGlobal/L1MPI/Doc2"
lasso_path_doc3 = "/content/drive/MyDrive/Memoire/DIORES/Models/V4/LassoGlobal/L1MPI/Doc3"

#### TESTE DE DIORES ENSEMBLISTE

In [14]:
# Créer le modèle ensembliste
ensemble_doc1 = DioresEnsembliste()
ensemble_doc2 = DioresEnsembliste()
ensemble_doc3 = DioresEnsembliste()


ensemble_doc1.load_models_from_files(model_paths_doc1, FEATURE_COLUMNS)
ensemble_doc2.load_models_from_files(model_paths_doc2, FEATURE_COLUMNS)
ensemble_doc3.load_models_from_files(model_paths_doc3, FEATURE_COLUMNS)

# Utilisation compatible avec evaluateV2
docs = [doc1_df_test, doc2_df_test, doc3_df_test]
predictors = [ensemble_doc1, ensemble_doc2, ensemble_doc3]

# Évaluation avec la même interface que DioresPredictorLasso
metrics = evaluateV2(docs, predictors, suffix='Ensembliste', rank_method='average')

# # Exemple d'utilisation directe
# df_test = pd.read_csv('/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc1/doc1_df_test.csv')
# df_result = ensemble.predict(df_test)
# print(df_result[['Score_Predit', 'Prediction_Status', 'Admission_Pred', 'Session_Pred', 'Mention_Pred']].head())


Modèles chargés avec succès
Modèles chargés avec succès
Modèles chargés avec succès

Ensembliste Doc1

Ensembliste Doc2

Ensembliste Doc3

EVALUATIONS POUR Ensembliste
rmse_Ensembliste_average: 19.0795
mae_Ensembliste_average: 14.4524
r2_Ensembliste_average: 0.0979
mrr_strict_Ensembliste_average: 3.2406
mrr_open_Ensembliste_average: 3.6104
precision_Ensembliste_average: 0.7661
recall_Ensembliste_average: 0.8214
f1_Ensembliste_average: 0.7926
equity_index_gender_Ensembliste_average: 1.8803
equity_index_region_Ensembliste_average: 4.7342

Analyse simplifiée de l'équité:

RMSE par genre:
  Doc1, Homme: RMSE = 16.4593 (n=54)
  Doc1, Femme: RMSE = 15.4990 (n=16)
  Doc2, Homme: RMSE = 20.0275 (n=53)
  Doc2, Femme: RMSE = 24.1128 (n=17)
  Doc3, Homme: RMSE = 21.4828 (n=50)
  Doc3, Femme: RMSE = 15.2463 (n=20)


#### TESTE DU MODÈLE DIORES LASSO

In [15]:
# Charger les modèles et créer les prédicteurs
directory = "/content/drive/MyDrive/Memoire/DIORES/Models/V4/LassoGlobal/L1MPI"
predictors = []

for i in range(1, 4):
    model_path = os.path.join(directory, f"Doc{i}")
    # Charger le modèle et le scaler
    model, scaler = load_model_and_scaler(model_path)

    # Charger les informations sur les features
    with open(os.path.join(model_path, 'lasso_globale_info.pkl'), 'rb') as f:
        info = pickle.load(f)

    # Créer le prédicteur avec les features spécifiques
    predictor = DioresPredictorLasso(model, scaler, info['features'])
    predictors.append(predictor)

print("DIORES Lasso")
# evaluate(docs, predictors, suffix='LassoGlobal', rank_method='average')
metrics = evaluateV2(docs, predictors, suffix='LassoGlobal', rank_method='average')
print("======================================================")

DIORES Lasso

LassoGlobal Doc1

LassoGlobal Doc2

LassoGlobal Doc3

EVALUATIONS POUR LassoGlobal
rmse_LassoGlobal_average: 17.5071
mae_LassoGlobal_average: 13.8619
r2_LassoGlobal_average: 0.2455
mrr_strict_LassoGlobal_average: 3.4502
mrr_open_LassoGlobal_average: 3.8350
precision_LassoGlobal_average: 0.0000
recall_LassoGlobal_average: 0.0000
f1_LassoGlobal_average: 0.0000
equity_index_gender_LassoGlobal_average: 2.0529
equity_index_region_LassoGlobal_average: 3.7664

Analyse simplifiée de l'équité:

RMSE par genre:
  Doc1, Homme: RMSE = 17.6002 (n=54)
  Doc1, Femme: RMSE = 18.4501 (n=16)
  Doc2, Homme: RMSE = 14.6037 (n=53)
  Doc2, Femme: RMSE = 19.2892 (n=17)
  Doc3, Homme: RMSE = 20.5409 (n=50)
  Doc3, Femme: RMSE = 13.7586 (n=20)


#### TESTE DU MODÈLE DIORES

In [16]:
# Créer le modèle hybride
hybrid_doc1 = DioresHybridPredictor()
hybrid_doc2 = DioresHybridPredictor()
hybrid_doc3 = DioresHybridPredictor()

# Charger les features Lasso depuis le fichier info
with open(os.path.join(lasso_path_doc1, 'lasso_globale_info.pkl'), 'rb') as f:
    lasso_info = pickle.load(f)
    LASSO_FEATURES_DOC1 = lasso_info['features']

with open(os.path.join(lasso_path_doc2, 'lasso_globale_info.pkl'), 'rb') as f:
    lasso_info = pickle.load(f)
    LASSO_FEATURES_DOC2 = lasso_info['features']

with open(os.path.join(lasso_path_doc3, 'lasso_globale_info.pkl'), 'rb') as f:
    lasso_info = pickle.load(f)
    LASSO_FEATURES_DOC3 = lasso_info['features']

hybrid_doc1.load_models_from_files(model_paths_doc1, lasso_path_doc1, FEATURE_COLUMNS, LASSO_FEATURES_DOC1)
hybrid_doc2.load_models_from_files(model_paths_doc2, lasso_path_doc2, FEATURE_COLUMNS, LASSO_FEATURES_DOC2)
hybrid_doc3.load_models_from_files(model_paths_doc3, lasso_path_doc3, FEATURE_COLUMNS, LASSO_FEATURES_DOC3)

# Utilisation avec evaluateV2
docs = [doc1_df_test, doc2_df_test, doc3_df_test]
predictors = [hybrid_doc1, hybrid_doc2, hybrid_doc3]
metrics = evaluateV2(docs, predictors, suffix='Hybrid', rank_method='average')

Tous les modèles chargés avec succès
Features Lasso (type: <class 'list'>): ['MATH', 'FR', 'PHILO', 'Age en Décembre 2018', 'S1', 'S2']
Tous les modèles chargés avec succès
Features Lasso (type: <class 'list'>): ['MATH', 'SCPH', 'SVT', 'PHILO', 'Moy. Gle', 'S1']
Tous les modèles chargés avec succès
Features Lasso (type: <class 'list'>): ['MATH', 'SCPH', 'FR', 'AN', 'Moy. sur Mat.Fond.', 'S1']

Hybrid Doc1
Colonnes DataFrame: ['REGION_DE_NAISSANCE', 'CREDIT', 'NIVEAU', 'SESSION', 'MENTION', 'MOYENNE ANNUELLE', 'RESULTAT', 'RESULTAT APP EVALUATION', 'Année BAC', 'Sexe', 'Série', 'Ets. de provenance', 'Type candidature', "Académie de l'Ets. Prov.", 'Résidence', "Centre d'Ec.", 'Nbre Fois au BAC', 'Mention', 'Résultat', 'Groupe Résultat', "Année de l'Extrait EC", 'Moy. nde', 'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR', 'PHILO', 'AN', 'SVT', 'COME', 'AFTA', 'EPAT', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle', 'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Score 

#### TESTE DU MODÈLE DAP

In [17]:
print("DAP")

predictor = DAP_predictor()
predictors = [predictor, predictor, predictor]

# evaluate(docs, predictors, suffix='DAP', rank_method='average')
metrics = evaluateV2(docs, predictors, suffix='DAP', rank_method='average')
print("======================================================")

DAP

DAP Doc1

DAP Doc2

DAP Doc3

EVALUATIONS POUR DAP
rmse_DAP_average: 15.8171
mae_DAP_average: 12.4905
r2_DAP_average: 0.3763
mrr_strict_DAP_average: 3.5985
mrr_open_DAP_average: 3.9516
precision_DAP_average: 0.0000
recall_DAP_average: 0.0000
f1_DAP_average: 0.0000
equity_index_gender_DAP_average: 2.0954
equity_index_region_DAP_average: 3.3363

Analyse simplifiée de l'équité:

RMSE par genre:
  Doc1, Homme: RMSE = 12.8936 (n=54)
  Doc1, Femme: RMSE = 16.2226 (n=16)
  Doc2, Homme: RMSE = 13.8331 (n=53)
  Doc2, Femme: RMSE = 18.2777 (n=17)
  Doc3, Homme: RMSE = 19.9366 (n=50)
  Doc3, Femme: RMSE = 15.1377 (n=20)


# PCSM TEST

In [18]:
doc1_df_test = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc1/doc1_df_test.csv");
doc2_df_test = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc2/doc2_df_test.csv");
doc3_df_test = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1PCSM/doc3/doc3_df_test.csv");

docs = []
docs.append( doc1_df_test)
docs.append( doc2_df_test)
docs.append( doc3_df_test)


model_paths_doc1 = {
    'admission': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1PCSM/Doc1/admi_non_admi_best_model.pkl',
    'session': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1PCSM/Doc1/session_best_model.pkl',
    'mention': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1PCSM/Doc1/mention_best_model.pkl'
}

model_paths_doc2 = {
    'admission': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1PCSM/Doc2/admi_non_admi_best_model.pkl',
    'session': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1PCSM/Doc2/session_best_model.pkl',
    'mention': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1PCSM/Doc2/mention_best_model.pkl'
}

model_paths_doc3 = {
    'admission': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1PCSM/Doc3/admi_non_admi_best_model.pkl',
    'session': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1PCSM/Doc3/session_best_model.pkl',
    'mention': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1PCSM/Doc3/mention_best_model.pkl'
}


lasso_path_doc1 = "/content/drive/MyDrive/Memoire/DIORES/Models/V4/LassoGlobal/L1PCSM/Doc1"
lasso_path_doc2 = "/content/drive/MyDrive/Memoire/DIORES/Models/V4/LassoGlobal/L1PCSM/Doc2"
lasso_path_doc3 = "/content/drive/MyDrive/Memoire/DIORES/Models/V4/LassoGlobal/L1PCSM/Doc3"

#### TESTE DE DIORES ENSEMBLISTE

In [19]:
# Créer le modèle ensembliste
ensemble_doc1 = DioresEnsembliste()
ensemble_doc2 = DioresEnsembliste()
ensemble_doc3 = DioresEnsembliste()


ensemble_doc1.load_models_from_files(model_paths_doc1, FEATURE_COLUMNS)
ensemble_doc2.load_models_from_files(model_paths_doc2, FEATURE_COLUMNS)
ensemble_doc3.load_models_from_files(model_paths_doc3, FEATURE_COLUMNS)

# Utilisation compatible avec evaluateV2
docs = [doc1_df_test, doc2_df_test, doc3_df_test]
predictors = [ensemble_doc1, ensemble_doc2, ensemble_doc3]

# Évaluation avec la même interface que DioresPredictorLasso
metrics = evaluateV2(docs, predictors, suffix='Ensembliste', rank_method='average')

Modèles chargés avec succès
Modèles chargés avec succès
Modèles chargés avec succès

Ensembliste Doc1

Ensembliste Doc2

Ensembliste Doc3

EVALUATIONS POUR Ensembliste
rmse_Ensembliste_average: 78.4413
mae_Ensembliste_average: 62.6272
r2_Ensembliste_average: -0.6590
mrr_strict_Ensembliste_average: 3.2194
mrr_open_Ensembliste_average: 3.6448
precision_Ensembliste_average: 0.5299
recall_Ensembliste_average: 0.5045
f1_Ensembliste_average: 0.5163
equity_index_gender_Ensembliste_average: 4.4745
equity_index_region_Ensembliste_average: 14.3804

Analyse simplifiée de l'équité:

RMSE par genre:
  Doc1, Homme: RMSE = 74.4447 (n=145)
  Doc1, Femme: RMSE = 90.6790 (n=66)
  Doc2, Homme: RMSE = 81.8831 (n=143)
  Doc2, Femme: RMSE = 71.5460 (n=68)
  Doc3, Homme: RMSE = 76.6738 (n=156)
  Doc3, Femme: RMSE = 76.9494 (n=55)


#### TESTE DU MODÈLE DIORES LASSO

In [20]:
# Charger les modèles et créer les prédicteurs
directory = "/content/drive/MyDrive/Memoire/DIORES/Models/V4/LassoGlobal/L1MPI"
predictors = []

for i in range(1, 4):
    model_path = os.path.join(directory, f"Doc{i}")
    # Charger le modèle et le scaler
    model, scaler = load_model_and_scaler(model_path)

    # Charger les informations sur les features
    with open(os.path.join(model_path, 'lasso_globale_info.pkl'), 'rb') as f:
        info = pickle.load(f)

    # Créer le prédicteur avec les features spécifiques
    predictor = DioresPredictorLasso(model, scaler, info['features'])
    predictors.append(predictor)

print("DIORES Lasso")
# evaluate(docs, predictors, suffix='LassoGlobal', rank_method='average')
metrics = evaluateV2(docs, predictors, suffix='LassoGlobal', rank_method='average')
print("======================================================")

DIORES Lasso

LassoGlobal Doc1

LassoGlobal Doc2

LassoGlobal Doc3

EVALUATIONS POUR LassoGlobal
rmse_LassoGlobal_average: 75.8781
mae_LassoGlobal_average: 61.3555
r2_LassoGlobal_average: -0.5519
mrr_strict_LassoGlobal_average: 3.4693
mrr_open_LassoGlobal_average: 3.9025
precision_LassoGlobal_average: 0.0000
recall_LassoGlobal_average: 0.0000
f1_LassoGlobal_average: 0.0000
equity_index_gender_LassoGlobal_average: 0.7278
equity_index_region_LassoGlobal_average: 10.2243

Analyse simplifiée de l'équité:

RMSE par genre:
  Doc1, Homme: RMSE = 76.1175 (n=145)
  Doc1, Femme: RMSE = 75.8845 (n=66)
  Doc2, Homme: RMSE = 74.7477 (n=143)
  Doc2, Femme: RMSE = 78.4257 (n=68)
  Doc3, Homme: RMSE = 75.5180 (n=156)
  Doc3, Femme: RMSE = 75.9740 (n=55)


#### TESTE DU MODÈLE DIORES

In [21]:
# Créer le modèle hybride
hybrid_doc1 = DioresHybridPredictor()
hybrid_doc2 = DioresHybridPredictor()
hybrid_doc3 = DioresHybridPredictor()

# Charger les features Lasso depuis le fichier info
with open(os.path.join(lasso_path_doc1, 'lasso_globale_info.pkl'), 'rb') as f:
    lasso_info = pickle.load(f)
    LASSO_FEATURES_DOC1 = lasso_info['features']

with open(os.path.join(lasso_path_doc2, 'lasso_globale_info.pkl'), 'rb') as f:
    lasso_info = pickle.load(f)
    LASSO_FEATURES_DOC2 = lasso_info['features']

with open(os.path.join(lasso_path_doc3, 'lasso_globale_info.pkl'), 'rb') as f:
    lasso_info = pickle.load(f)
    LASSO_FEATURES_DOC3 = lasso_info['features']

hybrid_doc1.load_models_from_files(model_paths_doc1, lasso_path_doc1, FEATURE_COLUMNS, LASSO_FEATURES_DOC1)
hybrid_doc2.load_models_from_files(model_paths_doc2, lasso_path_doc2, FEATURE_COLUMNS, LASSO_FEATURES_DOC2)
hybrid_doc3.load_models_from_files(model_paths_doc3, lasso_path_doc3, FEATURE_COLUMNS, LASSO_FEATURES_DOC3)

# Utilisation avec evaluateV2
docs = [doc1_df_test, doc2_df_test, doc3_df_test]
predictors = [hybrid_doc1, hybrid_doc2, hybrid_doc3]
metrics = evaluateV2(docs, predictors, suffix='Hybrid', rank_method='average')

Tous les modèles chargés avec succès
Features Lasso (type: <class 'list'>): ['MATH', 'SCPH', 'FR', 'AN', 'Moy. sur Mat.Fond.', 'Academie perf.']
Tous les modèles chargés avec succès
Features Lasso (type: <class 'list'>): ['MATH', 'SCPH', 'FR', 'PHILO', 'Age en Décembre 2018', 'S1']
Tous les modèles chargés avec succès
Features Lasso (type: <class 'list'>): ['MATH', 'SCPH', 'FR', 'PHILO', 'Moy. sur Mat.Fond.', 'Residence perf.']

Hybrid Doc1
Colonnes DataFrame: ['REGION_DE_NAISSANCE', 'CREDIT', 'NIVEAU', 'SESSION', 'MENTION', 'MOYENNE ANNUELLE', 'RESULTAT', 'RESULTAT APP EVALUATION', 'Année BAC', 'Sexe', 'Série', 'Ets. de provenance', 'Type candidature', "Académie de l'Ets. Prov.", 'Résidence', "Centre d'Ec.", 'Nbre Fois au BAC', 'Mention', 'Résultat', 'Groupe Résultat', "Année de l'Extrait EC", 'Moy. nde', 'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR', 'PHILO', 'AN', 'SVT', 'COME', 'AFTA', 'EPAT', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle', 'Moy. sur Mat.Fon

#### TESTE DU MODÈLE DAP

In [22]:
print("DAP")

predictor = DAP_predictor()
predictors = [predictor, predictor, predictor]

# evaluate(docs, predictors, suffix='DAP', rank_method='average')
metrics = evaluateV2(docs, predictors, suffix='DAP', rank_method='average')
print("======================================================")

DAP

DAP Doc1

DAP Doc2

DAP Doc3

EVALUATIONS POUR DAP
rmse_DAP_average: 69.9342
mae_DAP_average: 56.0411
r2_DAP_average: -0.3190
mrr_strict_DAP_average: 3.9604
mrr_open_DAP_average: 4.2579
precision_DAP_average: 0.0000
recall_DAP_average: 0.0000
f1_DAP_average: 0.0000
equity_index_gender_DAP_average: 2.9083
equity_index_region_DAP_average: 11.6017

Analyse simplifiée de l'équité:

RMSE par genre:
  Doc1, Homme: RMSE = 71.1865 (n=145)
  Doc1, Femme: RMSE = 69.5956 (n=66)
  Doc2, Homme: RMSE = 65.7816 (n=143)
  Doc2, Femme: RMSE = 71.3644 (n=68)
  Doc3, Homme: RMSE = 68.6578 (n=156)
  Doc3, Femme: RMSE = 78.9337 (n=55)


# BCGS TEST

In [23]:
doc1_df_test = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc1/doc1_df_test.csv");
doc2_df_test = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc2/doc2_df_test.csv");
doc3_df_test = pd.read_csv("/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1BCGS/doc3/doc3_df_test.csv");

docs = []
docs.append( doc1_df_test)
docs.append( doc2_df_test)
docs.append( doc3_df_test)



model_paths_doc1 = {
    'admission': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1BCGS/Doc1/admi_non_admi_best_model.pkl',
    'session': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1BCGS/Doc1/session_best_model.pkl',
    'mention': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1BCGS/Doc1/mention_best_model.pkl'
}

model_paths_doc2 = {
    'admission': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1BCGS/Doc2/admi_non_admi_best_model.pkl',
    'session': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1BCGS/Doc2/session_best_model.pkl',
    'mention': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1BCGS/Doc2/mention_best_model.pkl'
}

model_paths_doc3 = {
    'admission': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1BCGS/Doc3/admi_non_admi_best_model.pkl',
    'session': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1BCGS/Doc3/session_best_model.pkl',
    'mention': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1BCGS/Doc3/mention_best_model.pkl'
}


lasso_path_doc1 = "/content/drive/MyDrive/Memoire/DIORES/Models/V4/LassoGlobal/L1BCGS/Doc1"
lasso_path_doc2 = "/content/drive/MyDrive/Memoire/DIORES/Models/V4/LassoGlobal/L1BCGS/Doc2"
lasso_path_doc3 = "/content/drive/MyDrive/Memoire/DIORES/Models/V4/LassoGlobal/L1BCGS/Doc3"

#### TESTE DE DIORES ENSEMBLISTE

In [24]:
# Créer le modèle ensembliste
ensemble_doc1 = DioresEnsembliste()
ensemble_doc2 = DioresEnsembliste()
ensemble_doc3 = DioresEnsembliste()


ensemble_doc1.load_models_from_files(model_paths_doc1, FEATURE_COLUMNS)
ensemble_doc2.load_models_from_files(model_paths_doc2, FEATURE_COLUMNS)
ensemble_doc3.load_models_from_files(model_paths_doc3, FEATURE_COLUMNS)

# Utilisation compatible avec evaluateV2
docs = [doc1_df_test, doc2_df_test, doc3_df_test]
predictors = [ensemble_doc1, ensemble_doc2, ensemble_doc3]

# Évaluation avec la même interface que DioresPredictorLasso
metrics = evaluateV2(docs, predictors, suffix='Ensembliste', rank_method='average')

Modèles chargés avec succès
Modèles chargés avec succès
Modèles chargés avec succès

Ensembliste Doc1

Ensembliste Doc2

Ensembliste Doc3

EVALUATIONS POUR Ensembliste
rmse_Ensembliste_average: 58.1067
mae_Ensembliste_average: 47.4352
r2_Ensembliste_average: -0.6486
mrr_strict_Ensembliste_average: 3.6152
mrr_open_Ensembliste_average: 4.4116
precision_Ensembliste_average: 0.6929
recall_Ensembliste_average: 0.7912
f1_Ensembliste_average: 0.7387
equity_index_gender_Ensembliste_average: 0.8251
equity_index_region_Ensembliste_average: 8.9941

Analyse simplifiée de l'équité:

RMSE par genre:
  Doc1, Homme: RMSE = 59.0596 (n=75)
  Doc1, Femme: RMSE = 59.7248 (n=82)
  Doc2, Homme: RMSE = 54.2182 (n=77)
  Doc2, Femme: RMSE = 57.3815 (n=80)
  Doc3, Homme: RMSE = 58.5424 (n=85)
  Doc3, Femme: RMSE = 59.6643 (n=72)


#### TESTE DU MODÈLE DIORES LASSO

In [25]:
# Charger les modèles et créer les prédicteurs
directory = "/content/drive/MyDrive/Memoire/DIORES/Models/V4/LassoGlobal/L1MPI"
predictors = []

for i in range(1, 4):
    model_path = os.path.join(directory, f"Doc{i}")
    # Charger le modèle et le scaler
    model, scaler = load_model_and_scaler(model_path)

    # Charger les informations sur les features
    with open(os.path.join(model_path, 'lasso_globale_info.pkl'), 'rb') as f:
        info = pickle.load(f)

    # Créer le prédicteur avec les features spécifiques
    predictor = DioresPredictorLasso(model, scaler, info['features'])
    predictors.append(predictor)

print("DIORES Lasso")
# evaluate(docs, predictors, suffix='LassoGlobal', rank_method='average')
metrics = evaluateV2(docs, predictors, suffix='LassoGlobal', rank_method='average')
print("======================================================")

DIORES Lasso

LassoGlobal Doc1

LassoGlobal Doc2

LassoGlobal Doc3

EVALUATIONS POUR LassoGlobal
rmse_LassoGlobal_average: 59.0729
mae_LassoGlobal_average: 47.5350
r2_LassoGlobal_average: -0.7042
mrr_strict_LassoGlobal_average: 3.5116
mrr_open_LassoGlobal_average: 4.0907
precision_LassoGlobal_average: 0.0000
recall_LassoGlobal_average: 0.0000
f1_LassoGlobal_average: 0.0000
equity_index_gender_LassoGlobal_average: 3.3733
equity_index_region_LassoGlobal_average: 7.4082

Analyse simplifiée de l'équité:

RMSE par genre:
  Doc1, Homme: RMSE = 62.6387 (n=75)
  Doc1, Femme: RMSE = 60.0345 (n=82)
  Doc2, Homme: RMSE = 63.5269 (n=77)
  Doc2, Femme: RMSE = 49.5753 (n=80)
  Doc3, Homme: RMSE = 60.7401 (n=85)
  Doc3, Femme: RMSE = 57.0559 (n=72)


#### TESTE DU MODÈLE DIORES

In [26]:
# Créer le modèle hybride
hybrid_doc1 = DioresHybridPredictor()
hybrid_doc2 = DioresHybridPredictor()
hybrid_doc3 = DioresHybridPredictor()

# Charger les features Lasso depuis le fichier info
with open(os.path.join(lasso_path_doc1, 'lasso_globale_info.pkl'), 'rb') as f:
    lasso_info = pickle.load(f)
    LASSO_FEATURES_DOC1 = lasso_info['features']

with open(os.path.join(lasso_path_doc2, 'lasso_globale_info.pkl'), 'rb') as f:
    lasso_info = pickle.load(f)
    LASSO_FEATURES_DOC2 = lasso_info['features']

with open(os.path.join(lasso_path_doc3, 'lasso_globale_info.pkl'), 'rb') as f:
    lasso_info = pickle.load(f)
    LASSO_FEATURES_DOC3 = lasso_info['features']

hybrid_doc1.load_models_from_files(model_paths_doc1, lasso_path_doc1, FEATURE_COLUMNS, LASSO_FEATURES_DOC1)
hybrid_doc2.load_models_from_files(model_paths_doc2, lasso_path_doc2, FEATURE_COLUMNS, LASSO_FEATURES_DOC2)
hybrid_doc3.load_models_from_files(model_paths_doc3, lasso_path_doc3, FEATURE_COLUMNS, LASSO_FEATURES_DOC3)

# Utilisation avec evaluateV2
docs = [doc1_df_test, doc2_df_test, doc3_df_test]
predictors = [hybrid_doc1, hybrid_doc2, hybrid_doc3]
metrics = evaluateV2(docs, predictors, suffix='Hybrid', rank_method='average')

Tous les modèles chargés avec succès
Features Lasso (type: <class 'list'>): ['SCPH', 'SVT', 'Moy. Gle', 'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'S2']
Tous les modèles chargés avec succès
Features Lasso (type: <class 'list'>): ['SCPH', 'SVT', 'FR', 'PHILO', 'AN', 'Academie perf.']
Tous les modèles chargés avec succès
Features Lasso (type: <class 'list'>): ['MATH', 'SVT', 'FR', 'PHILO', 'AN', 'S1']

Hybrid Doc1
Colonnes DataFrame: ['REGION_DE_NAISSANCE', 'CREDIT', 'NIVEAU', 'SESSION', 'MENTION', 'MOYENNE ANNUELLE', 'RESULTAT', 'RESULTAT APP EVALUATION', 'Année BAC', 'Sexe', 'Série', 'Ets. de provenance', 'Type candidature', "Académie de l'Ets. Prov.", 'Résidence', "Centre d'Ec.", 'Nbre Fois au BAC', 'Mention', 'Résultat', 'Groupe Résultat', "Année de l'Extrait EC", 'Moy. nde', 'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR', 'PHILO', 'AN', 'SVT', 'COME', 'AFTA', 'EPAT', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle', 'Moy. sur Mat.Fond.', 'Age en Décembre 2018

#### TESTE DU MODÈLE DAP

In [27]:
print("DAP")

predictor = DAP_predictor()
predictors = [predictor, predictor, predictor]

# evaluate(docs, predictors, suffix='DAP', rank_method='average')
metrics = evaluateV2(docs, predictors, suffix='DAP', rank_method='average')
print("======================================================")

DAP

DAP Doc1

DAP Doc2

DAP Doc3

EVALUATIONS POUR DAP
rmse_DAP_average: 55.6465
mae_DAP_average: 44.1295
r2_DAP_average: -0.5124
mrr_strict_DAP_average: 4.0911
mrr_open_DAP_average: 4.4992
precision_DAP_average: 0.0000
recall_DAP_average: 0.0000
f1_DAP_average: 0.0000
equity_index_gender_DAP_average: 2.0291
equity_index_region_DAP_average: 11.5435

Analyse simplifiée de l'équité:

RMSE par genre:
  Doc1, Homme: RMSE = 58.7412 (n=75)
  Doc1, Femme: RMSE = 57.1533 (n=82)
  Doc2, Homme: RMSE = 56.3991 (n=77)
  Doc2, Femme: RMSE = 54.5755 (n=80)
  Doc3, Homme: RMSE = 57.3852 (n=85)
  Doc3, Femme: RMSE = 48.6220 (n=72)


In [28]:
# Méthode 2: Charger directement depuis Google Colab (si vous êtes sur Colab)
def load_models_from_colab_drive(drive_paths):
    """
    Charge les modèles depuis Drive monté sur Google Colab

    Args:
        drive_paths: dict avec les chemins Drive
                    {'admission': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc1/admi_non_admi_best_model_LinearSVC.pkl'
                     'session': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc1/session_best_model_LinearSVC.pkl',
                     'mention':  '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc1/mention_best_model_LinearSVC.pkl', }
    """
    from google.colab import drive

    # Monter Google Drive
    drive.mount('/content/drive')

    models = {}
    for name, path in drive_paths.items():
        with open(path, 'rb') as f:
            models[name] = pickle.load(f)
        print(f"Modèle {name} chargé avec succès depuis Drive")
    return models

In [29]:
# Fonction principale de test (version Drive)
def test_models_from_drive(drive_source, test_data_path, feature_columns, method='download'):
    """
    Teste les trois modèles depuis Google Drive

    Args:
        drive_source: URLs ou chemins Drive selon la méthode
        test_data_path: chemin vers le dataset de test
        feature_columns: liste des colonnes à utiliser comme features
        method: 'download' (gdown) ou 'colab' (drive monté)
    """

    # Charger les modèles selon la méthode choisie
    if method == 'download':
        print("Téléchargement des modèles depuis Google Drive...")
        model_paths = download_models_from_drive(drive_source)
        models = load_models(model_paths)
    elif method == 'colab':
        print("Chargement des modèles depuis Drive monté...")
        models = load_models_from_colab_drive(drive_source)
    else:
        raise ValueError("Méthode doit être 'download' ou 'colab'")

    # Charger le dataset de test
    print("Chargement du dataset de test...")
    df_test = pd.read_csv(test_data_path)  # ou pd.read_excel() selon le format
    print(f"Dataset de test chargé: {df_test.shape[0]} échantillons, {df_test.shape[1]} colonnes")

    # Préparer les features
    X_test = df_test[feature_columns]

    # Définir les labels pour chaque modèle selon votre encodage
    class_labels_dict = {
        'admission': ['NON ADMIS', 'AUTORISE/PASSE'],
        'session': ['Deuxième Session', 'Première Session'],
        'mention': ['Passable', 'Assez-Bien/Bien/Très-Bien']
    }

    # Variables pour stocker les résultats
    results = {}

    # 1. Test du modèle d'admission
    if 'admission_target' in df_test.columns:  # Remplacez par le nom réel de votre colonne
        print("\n" + "="*60)
        print("TEST DU MODÈLE D'ADMISSION")
        print("="*60)
        y_test_admission = df_test['admission_target']  # Remplacez par le nom réel
        y_pred_admission, acc_admission = evaluate_model(
            models['admission'], X_test, y_test_admission,
            'Modèle d\'Admission', class_labels_dict['admission']
        )
        results['admission'] = {'predictions': y_pred_admission, 'accuracy': acc_admission}

    # 2. Test du modèle de session
    if 'session_target' in df_test.columns:  # Remplacez par le nom réel de votre colonne
        print("\n" + "="*60)
        print("TEST DU MODÈLE DE SESSION")
        print("="*60)
        y_test_session = df_test['session_target']  # Remplacez par le nom réel
        y_pred_session, acc_session = evaluate_model(
            models['session'], X_test, y_test_session,
            'Modèle de Session', class_labels_dict['session']
        )
        results['session'] = {'predictions': y_pred_session, 'accuracy': acc_session}

    # 3. Test du modèle de mention
    if 'mention_target' in df_test.columns:  # Remplacez par le nom réel de votre colonne
        print("\n" + "="*60)
        print("TEST DU MODÈLE DE MENTION")
        print("="*60)
        y_test_mention = df_test['mention_target']  # Remplacez par le nom réel
        y_pred_mention, acc_mention = evaluate_model(
            models['mention'], X_test, y_test_mention,
            'Modèle de Mention', class_labels_dict['mention']
        )
        results['mention'] = {'predictions': y_pred_mention, 'accuracy': acc_mention}

    # Résumé des performances
    print("\n" + "="*60)
    print("RÉSUMÉ DES PERFORMANCES")
    print("="*60)
    for model_name, result in results.items():
        print(f"{model_name.capitalize()}: {result['accuracy']:.4f} ({result['accuracy']*100:.2f}%)")

    return results

In [30]:
from sklearn.base import BaseEstimator, ClassifierMixin
import numpy as np
import pandas as pd
import pickle

class DioresEnsembliste(BaseEstimator, ClassifierMixin):
    """
    Modèle ensembliste hiérarchique pour la prédiction des résultats étudiants.

    Architecture en arbre :
    1. Admission (NON ADMIS/AUTORISE/PASSE)
    2. Si admis → Session (Deuxième/Première)
    3. Si Première Session → Mention (Passable/Assez-Bien/Bien/Très-Bien)

    Résultat final : dictionnaire avec toutes les prédictions
    """

    def __init__(self, model_admission=None, model_session=None, model_mention=None):
        """
        Initialise le modèle ensembliste

        Args:
            model_admission: modèle pour prédire l'admission
            model_session: modèle pour prédire la session
            model_mention: modèle pour prédire la mention
        """
        self.model_admission = model_admission
        self.model_session = model_session
        self.model_mention = model_mention

        # Mappings pour convertir les prédictions numériques en labels
        self.admission_labels = {0: 'NON ADMIS', 1: 'AUTORISE/PASSE'}
        self.session_labels = {0: 'Deuxième Session', 1: 'Première Session'}
        self.mention_labels = {0: 'Passable', 1: 'Assez-Bien/Bien/Très-Bien'}

    def load_models_from_files(self, model_paths):
        """
        Charge les modèles depuis des fichiers pickle

        Args:
            model_paths: dict avec les chemins des modèles
                        {'admission': 'path1.pkl', 'session': 'path2.pkl', 'mention': 'path3.pkl'}
        """
        if 'admission' in model_paths:
            with open(model_paths['admission'], 'rb') as f:
                self.model_admission = pickle.load(f)

        if 'session' in model_paths:
            with open(model_paths['session'], 'rb') as f:
                self.model_session = pickle.load(f)

        if 'mention' in model_paths:
            with open(model_paths['mention'], 'rb') as f:
                self.model_mention = pickle.load(f)

        print("Modèles chargés avec succès")
        return self

    def fit(self, X, y):
        """
        Méthode fit (requise par BaseEstimator)
        Dans notre cas, les modèles sont déjà entraînés
        """
        # Les modèles individuels sont déjà entraînés
        # Cette méthode est juste pour la compatibilité sklearn
        return self

    def predict_single(self, x):
        """
        Prédit le parcours complet d'un seul étudiant

        Args:
            x: features d'un étudiant (array 1D)

        Returns:
            dict: résultat complet du parcours
        """
        result = {
            'admission': None,
            'session': None,
            'mention': None,
            'final_status': None,
            'path': []
        }

        # Reshape pour avoir la bonne forme (1, n_features)
        x_reshaped = x.reshape(1, -1)

        # Étape 1: Prédiction de l'admission
        if self.model_admission is not None:
            admission_pred = self.model_admission.predict(x_reshaped)[0]
            admission_label = self.admission_labels[admission_pred]
            result['admission'] = admission_label
            result['path'].append(f"Admission: {admission_label}")

            # Si NON ADMIS, on s'arrête ici
            if admission_pred == 0:  # NON ADMIS
                result['final_status'] = 'NON ADMIS'
                result['path'].append("ARRÊT: Non admis")
                return result

            # Si AUTORISE/PASSE, on continue avec la session
            if self.model_session is not None:
                session_pred = self.model_session.predict(x_reshaped)[0]
                session_label = self.session_labels[session_pred]
                result['session'] = session_label
                result['path'].append(f"Session: {session_label}")

                # Si Deuxième Session, on s'arrête ici
                if session_pred == 0:  # Deuxième Session
                    result['final_status'] = f"AUTORISE - {session_label}"
                    result['path'].append("ARRÊT: Deuxième session")
                    return result

                # Si Première Session, on continue avec la mention
                if self.model_mention is not None:
                    mention_pred = self.model_mention.predict(x_reshaped)[0]
                    mention_label = self.mention_labels[mention_pred]
                    result['mention'] = mention_label
                    result['path'].append(f"Mention: {mention_label}")
                    result['final_status'] = f"AUTORISE - Première Session - {mention_label}"
                    result['path'].append("ARRÊT: Parcours complet")
                else:
                    result['final_status'] = f"AUTORISE - {session_label}"
                    result['path'].append("ARRÊT: Pas de modèle mention")
            else:
                result['final_status'] = admission_label
                result['path'].append("ARRÊT: Pas de modèle session")
        else:
            result['final_status'] = "ERREUR: Pas de modèle admission"
            result['path'].append("ERREUR: Pas de modèle admission")

        return result

    def predict(self, X):
        """
        Prédit pour un ensemble d'étudiants

        Args:
            X: features des étudiants (array 2D)

        Returns:
            list: liste des résultats pour chaque étudiant
        """
        results = []
        for i in range(len(X)):
            result = self.predict_single(X[i])
            results.append(result)

        return results

    def predict_final_status(self, X):
        """
        Retourne seulement le statut final pour chaque étudiant

        Args:
            X: features des étudiants

        Returns:
            list: liste des statuts finaux
        """
        results = self.predict(X)
        return [result['final_status'] for result in results]

    def predict_admission_only(self, X):
        """
        Retourne seulement les prédictions d'admission (compatible sklearn)

        Args:
            X: features des étudiants

        Returns:
            array: prédictions d'admission (0 ou 1)
        """
        if self.model_admission is None:
            raise ValueError("Modèle d'admission non chargé")

        return self.model_admission.predict(X)

    def analyze_student_path(self, X, student_index=0):
        """
        Analyse détaillée du parcours d'un étudiant spécifique

        Args:
            X: features des étudiants
            student_index: index de l'étudiant à analyser
        """
        result = self.predict_single(X[student_index])

        print(f"ANALYSE DE L'ÉTUDIANT {student_index}")
        print("="*50)

        for step in result['path']:
            print(f"• {step}")

        print(f"\nSTATUT FINAL: {result['final_status']}")

        return result

    def get_statistics(self, X):
        """
        Calcule des statistiques sur les prédictions

        Args:
            X: features des étudiants

        Returns:
            dict: statistiques détaillées
        """
        results = self.predict(X)

        stats = {
            'total_students': len(results),
            'non_admis': 0,
            'admis_deuxieme_session': 0,
            'admis_premiere_session': 0,
            'mentions': {'Passable': 0, 'Assez-Bien/Bien/Très-Bien': 0}
        }

        for result in results:
            if 'NON ADMIS' in result['final_status']:
                stats['non_admis'] += 1
            elif 'Deuxième Session' in result['final_status']:
                stats['admis_deuxieme_session'] += 1
            elif 'Première Session' in result['final_status']:
                stats['admis_premiere_session'] += 1

                if result['mention']:
                    if 'Passable' in result['mention']:
                        stats['mentions']['Passable'] += 1
                    else:
                        stats['mentions']['Assez-Bien/Bien/Très-Bien'] += 1

        # Calculer les pourcentages
        total = stats['total_students']
        stats['percentages'] = {
            'non_admis': (stats['non_admis'] / total) * 100,
            'admis_deuxieme_session': (stats['admis_deuxieme_session'] / total) * 100,
            'admis_premiere_session': (stats['admis_premiere_session'] / total) * 100
        }

        return stats

    def print_statistics(self, X):
        """Affiche les statistiques de manière lisible"""
        stats = self.get_statistics(X)

        print("STATISTIQUES DES PRÉDICTIONS")
        print("="*40)
        print(f"Total étudiants: {stats['total_students']}")
        print(f"Non admis: {stats['non_admis']} ({stats['percentages']['non_admis']:.1f}%)")
        print(f"Admis 2ème session: {stats['admis_deuxieme_session']} ({stats['percentages']['admis_deuxieme_session']:.1f}%)")
        print(f"Admis 1ère session: {stats['admis_premiere_session']} ({stats['percentages']['admis_premiere_session']:.1f}%)")

        if stats['admis_premiere_session'] > 0:
            print("\nMentions (1ère session):")
            print(f"  Passable: {stats['mentions']['Passable']}")
            print(f"  Assez-Bien/Bien/Très-Bien: {stats['mentions']['Assez-Bien/Bien/Très-Bien']}")

# Exemple d'utilisation
if __name__ == "__main__":

    # Colonnes de features
    FEATURE_COLUMNS = ['Année BAC', 'Nbre Fois au BAC', 'Groupe Résultat', 'Moy. nde',
                      'Moy. ère', 'Moy. S Term.', 'Moy. S Term..1', 'MATH', 'SCPH', 'FR',
                      'PHILO', 'AN', 'Tot. Pts au Grp.', 'Moyenne au Grp.', 'Moy. Gle',
                      'Moy. sur Mat.Fond.', 'Age en Décembre 2018', 'Série_Encode',
                      'Sexe_Encode', 'Academie perf.', 'Résidence_Encode',
                      'Ets. de provenance_Encode', 'Centre d\'Ec._Encode',
                      'Académie de l\'Ets. Prov._Encode', 'REGION_DE_NAISSANCE_Encode']

    # Créer le modèle ensembliste
    ensemble = DioresEnsembliste()

    # Charger les modèles depuis les fichiers
    drive_paths = {
    'admission': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc1/admi_non_admi_best_model_LinearSVC.pkl',
    'session': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc1/session_best_model_LinearSVC.pkl',
    'mention': '/content/drive/MyDrive/Memoire/DIORES/Models/V4/Classifiers/L1MPI/Doc1/mention_best_model_LinearSVC.pkl'
    }

    ensemble.load_models_from_files(drive_paths)


    # Utilisation avec des données de test
    df_test = pd.read_csv('/content/drive/MyDrive/Memoire/DIORES/Datasets/V2/L1MPI/doc1/doc1_df_test.csv')
    X_test = df_test[FEATURE_COLUMNS].values

    # Prédictions complètes
    results = ensemble.predict(X_test)

    # Analyse d'un étudiant spécifique
    ensemble.analyze_student_path(X_test, student_index=0)

    # Statistiques globales
    ensemble.print_statistics(X_test)

Modèles chargés avec succès
ANALYSE DE L'ÉTUDIANT 0
• Admission: AUTORISE/PASSE
• Session: Deuxième Session
• ARRÊT: Deuxième session

STATUT FINAL: AUTORISE - Deuxième Session
STATISTIQUES DES PRÉDICTIONS
Total étudiants: 70
Non admis: 25 (35.7%)
Admis 2ème session: 9 (12.9%)
Admis 1ère session: 36 (51.4%)

Mentions (1ère session):
  Passable: 25
  Assez-Bien/Bien/Très-Bien: 11
